<a href="https://colab.research.google.com/github/NMapelu/AirlineChatbot/blob/main/notebooks/03_train_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

DRIVE_ROOT = "/content/drive/MyDrive/AirlineChatbot"
PROCESSED_DIR = f"{DRIVE_ROOT}/data/processed"
MODELS_DIR = f"{DRIVE_ROOT}/outputs/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Timestamped run folder so multiple runs don't overwrite each other
RUN_ID = datetime.now().strftime("baseline_%Y-%m-%d_%H-%M-%S")
RUN_DIR = f"{MODELS_DIR}/{RUN_ID}"
os.makedirs(RUN_DIR, exist_ok=True)

print("Processed data:", PROCESSED_DIR)
print("Run directory :", RUN_DIR)

Processed data: /content/drive/MyDrive/AirlineChatbot/data/processed
Run directory : /content/drive/MyDrive/AirlineChatbot/outputs/models/baseline_2026-09-18_01-42-33


In [3]:
train_df = pd.read_csv(f"{PROCESSED_DIR}/train.csv")
val_df   = pd.read_csv(f"{PROCESSED_DIR}/val.csv")
test_df  = pd.read_csv(f"{PROCESSED_DIR}/test.csv")

with open(f"{PROCESSED_DIR}/label_map.json") as f:
    label_map = json.load(f)

# Convert string keys back to int (JSON stores them as strings)
label_map = {int(k): v for k, v in label_map.items()}

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("Classes:", len(label_map))

# Extract arrays
X_train, y_train = train_df["text"].tolist(), train_df["label"].tolist()
X_val,   y_val   = val_df["text"].tolist(),   val_df["label"].tolist()
X_test,  y_test  = test_df["text"].tolist(),  test_df["label"].tolist()

Train: (24480, 3)
Val  : (3060, 3)
Test : (3061, 3)
Classes: 33


In [4]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),      # unigrams + bigrams
        min_df=2,                # ignore very rare terms
        max_df=0.95,             # ignore near-ubiquitous terms
        sublinear_tf=True,       # dampen term frequency skew
        lowercase=True,
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        C=1.0,
        n_jobs=-1,
        multi_class="multinomial",
    )),
])

print("Pipeline ready.")

Pipeline ready.


In [5]:
import time

start = time.time()
pipeline.fit(X_train, y_train)
elapsed = time.time() - start

print(f"Training complete in {elapsed:.2f} seconds")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training complete in 4.40 seconds


In [6]:
val_preds = pipeline.predict(X_val)

val_acc = accuracy_score(y_val, val_preds)
val_f1  = f1_score(y_val, val_preds, average="macro")

print(f"Validation accuracy : {val_acc:.4f}")
print(f"Validation macro-F1 : {val_f1:.4f}")

Validation accuracy : 0.9784
Validation macro-F1 : 0.9786


In [7]:
test_preds = pipeline.predict(X_test)

test_acc = accuracy_score(y_test, test_preds)
test_f1  = f1_score(y_test, test_preds, average="macro")

print(f"Test accuracy : {test_acc:.4f}")
print(f"Test macro-F1 : {test_f1:.4f}")

Test accuracy : 0.9762
Test macro-F1 : 0.9762


In [8]:
# Build target names in label order
target_names = [label_map[i] for i in range(len(label_map))]

report = classification_report(
    y_test, test_preds,
    target_names=target_names,
    digits=3,
    zero_division=0,
)
print(report)

                                 precision    recall  f1-score   support

                    book_flight      1.000     0.968     0.984        95
                      book_trip      0.920     0.920     0.920        88
                  cancel_flight      1.000     0.990     0.995        99
                    cancel_trip      1.000     0.989     0.995        93
                  change_flight      0.989     0.959     0.974        97
                    change_seat      0.989     0.967     0.978        90
                    change_trip      0.951     1.000     0.975        97
             check_arrival_time      1.000     0.987     0.993        77
        check_baggage_allowance      1.000     1.000     1.000        94
         check_cancellation_fee      1.000     1.000     1.000        97
           check_departure_time      1.000     1.000     1.000        76
check_flight_insurance_coverage      0.959     0.989     0.974        95
            check_flight_offers      1.000     0.9

In [9]:
cm = confusion_matrix(y_test, test_preds)

# Find the top 15 misclassifications (excluding the diagonal)
misclass = []
for i in range(len(cm)):
    for j in range(len(cm)):
        if i != j and cm[i, j] > 0:
            misclass.append((cm[i, j], label_map[i], label_map[j]))

misclass.sort(reverse=True)

print("Top misclassifications (true → predicted : count):")
print("-" * 60)
for count, true_label, pred_label in misclass[:15]:
    print(f"  {true_label:35s} → {pred_label:35s} : {count}")

Top misclassifications (true → predicted : count):
------------------------------------------------------------
  book_trip                           → search_trip                         : 7
  search_trip_insurance               → search_trip                         : 6
  print_boarding_pass                 → get_boarding_pass                   : 6
  purchase_trip_insurance             → search_trip_insurance               : 5
  search_trip_insurance               → purchase_trip_insurance             : 4
  search_trip                         → book_trip                           : 4
  check_trip_insurance_coverage       → check_flight_insurance_coverage     : 4
  search_flight_insurance             → search_flight                       : 3
  check_flight_prices                 → search_flight                       : 3
  get_refund                          → search_trip                         : 2
  check_flight_offers                 → check_flight_reservation            : 2
  change

In [10]:
samples = [
    "I want to book a flight to London next Friday",
    "Can I bring two suitcases on my trip?",
    "How much does it cost to change my seat?",
    "Please cancel my reservation",
    "I need a human agent right now",
]

preds = pipeline.predict(samples)

print("Predictions on sample inputs:")
print("=" * 70)
for text, pred in zip(samples, preds):
    print(f"  '{text}'")
    print(f"     → {label_map[pred]}")
    print()

Predictions on sample inputs:
  'I want to book a flight to London next Friday'
     → book_flight

  'Can I bring two suitcases on my trip?'
     → cancel_trip

  'How much does it cost to change my seat?'
     → change_seat

  'Please cancel my reservation'
     → cancel_flight

  'I need a human agent right now'
     → human_agent



In [11]:
# Save the pipeline
model_path = f"{RUN_DIR}/model.joblib"
joblib.dump(pipeline, model_path)

# Save label map
with open(f"{RUN_DIR}/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

# Save metrics
metrics = {
    "run_id": RUN_ID,
    "train_size": len(X_train),
    "val_size": len(X_val),
    "test_size": len(X_test),
    "num_classes": len(label_map),
    "training_seconds": round(elapsed, 2),
    "val_accuracy": round(val_acc, 4),
    "val_macro_f1": round(val_f1, 4),
    "test_accuracy": round(test_acc, 4),
    "test_macro_f1": round(test_f1, 4),
    "model_type": "TF-IDF + Logistic Regression",
}

with open(f"{RUN_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save classification report as text
with open(f"{RUN_DIR}/classification_report.txt", "w") as f:
    f.write(report)

print("Saved artifacts:")
for fname in sorted(os.listdir(RUN_DIR)):
    fpath = f"{RUN_DIR}/{fname}"
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname}  ({size_kb:.1f} KB)")

Saved artifacts:
  classification_report.txt  (2.6 KB)
  label_map.json  (0.9 KB)
  metrics.json  (0.3 KB)
  model.joblib  (1238.2 KB)


In [12]:
print("=" * 60)
print("BASELINE TRAINING COMPLETE")
print("=" * 60)
print(f"Run ID          : {RUN_ID}")
print(f"Model           : TF-IDF + Logistic Regression")
print(f"Training time   : {elapsed:.2f}s")
print()
print(f"Validation acc  : {val_acc:.4f}")
print(f"Validation F1   : {val_f1:.4f}")
print(f"Test accuracy   : {test_acc:.4f}")
print(f"Test macro-F1   : {test_f1:.4f}")
print()
print(f"Artifacts saved : {RUN_DIR}")
print()
print("Next steps:")
print("  • Option A: Improve baseline (better vectorizer, hyperparameter tuning)")
print("  • Option B: Upgrade to DistilBERT for higher accuracy")
print("  • Option C: Build the chatbot loop using this classifier + response lookup")

BASELINE TRAINING COMPLETE
Run ID          : baseline_2026-09-18_01-42-33
Model           : TF-IDF + Logistic Regression
Training time   : 4.40s

Validation acc  : 0.9784
Validation F1   : 0.9786
Test accuracy   : 0.9762
Test macro-F1   : 0.9762

Artifacts saved : /content/drive/MyDrive/AirlineChatbot/outputs/models/baseline_2026-09-18_01-42-33

Next steps:
  • Option A: Improve baseline (better vectorizer, hyperparameter tuning)
  • Option B: Upgrade to DistilBERT for higher accuracy
  • Option C: Build the chatbot loop using this classifier + response lookup
